In [5]:
import math
import numpy as np
import matplotlib.pyplot as plt

### Engine

In [ ]:
class Value:
    def __init__(self, data, _prev=()):
        self.data = data
        self.grad = 0.0
        self._prev = set(_prev)
        self._backward = lambda: None
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        # closure  
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __neg__(self):
        out = Value(-self.data, (self,))

        def _backward():
            self.grad += -1 * out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        return self + (-other)
    
    def __pow__(self, other):
        assert isinstance (other, (int, float))
        out = Value((self.data ** other), (self,))

        def _backward():
            self.grad += out.grad * other * (self.data ** (other - 1))
        out._backward = _backward

        return out
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad 
        out._backward = _backward
        return out 
    
    def log(self):
        out = Value(math.log(self.data), (self,))

        def _backward():
            self.grad += (1/self.data) * out.grad 
        out._backward = _backward
        return out
    
    def sigmoid(self):
        s = 1 / (1 + (math.exp(-self.data)))
        out = Value(s, (self,))

        def _backward():
            self.grad += out.grad * (s * (1 - s))
        out._backward = _backward
        return out

    
    def relu(self):
        out = Value(self.data if self.data > 0 else 0, (self,))

        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out 

    def backward(self):
        topo = []
        visited = set()
        # Build a topological sort
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        self.grad = 1.0
        # call _backward() in reversed order
        for node in reversed(topo):
            node._backward()
    
    def __radd__(self, other):
        return self + other

    def __rsub__(self, other):
        return (-self) + other

    def __rmul__(self, other):
        return self * other

    def __rtruediv__(self, other):
        return other * (self ** -1)
    

    def __repr__(self):
        return f"Value: {self.data}"

In [7]:
s = 1 / (1 + (math.e ** -100))
s

1.0

In [8]:
a = Value(10)
b = 20

c = b / a
c

Value: 2.0

In [9]:
# Test for _backward
a = Value(10)
b = Value(5)

c = a + b
print(c)

c.grad = 1.0
c._backward()

print(a.grad)
print(b.grad)

Value: 15
1.0
1.0


In [10]:
# Test for _backward in __mul__
a = Value(10)
b = Value(5)

c = a * b
print(c)

c.grad = 1.0
print(c.grad)
print(a.grad)
print(b.grad)

c._backward()

print(a.grad)
print(b.grad)

Value: 50
1.0
0.0
0.0
5.0
10.0


In [11]:
# Testing backward()
a = Value(10)
b = Value(5)
d = Value(7)
c = (a * b) 
print(c)
print(c.grad)
c.backward()
print(c.grad)
print(a.grad)
print(b.grad)

Value: 50
0.0
1.0
5.0
10.0


### Neural Network

In [12]:
import random

class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def parameters(self):
        return []
    
class Linear(Module):
    def __init__(self, nin, nout):
        # w.shape = (nout, nin)
        self.w = [
            [Value(random.uniform(-0.1, 0.1)) for _ in range(nin)]
            for _ in range(nout)
        ]
        self.b = [Value(0) for _ in range(nout)]
    
    def __call__(self, x):
        out = []
        for w_row, b in zip(self.w, self.b):
            act = sum((wi * xi for wi, xi in zip(w_row, x)), b)
            out.append(act)
        if len(out) == 1:
            return out[0]
        return out
    
    def parameters(self):
        return [p for w_row in self.w for p in w_row] + self.b
    
    def __repr__(self):
        return f"Linear: ({len(self.w[0])}, {len(self.w)})"

class ReLU(Module):
    def __call__(self, x):
        if isinstance(x, list):
            return [xi.relu() for xi in x]
        return x.relu()
    
    def __repr__(self):
        return f"ReLU()"

class Sigmoid(Module):
    def __call__(self, x):
        if isinstance(x, list):
            return [xi.sigmoid() for xi in x]
        return x.sigmoid()
    
    def __repr__(self):
        return f"Sigmoid()"

class Sequential(Module):
    def __init__(self, *layers):
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
    def __repr__(self):
        layers = ",\n    ".join(str(l) for l in self.layers)
        n_params = len(self.parameters())

        return (
            f"Sequential(\n"
            f"    {layers}\n"
            f")\n\n"
            f"Parameters: {n_params}"
        )



In [13]:

def binary_cross_entropy(y_pred, y_true):
    eps = 1e-8
    l = -(y_true * y_pred.log() + (1 - y_true) * (1 - y_pred + eps).log())
    return l

In [14]:
# Test 
model = Sequential(
    Linear(3, 4),
    ReLU(), 
    Linear(4, 4),
    ReLU(),
    Linear(4, 1),
    Sigmoid()
)
x = [Value(3),Value(5), Value(10)]
y_true = 0
y_pred = model(x)

loss = binary_cross_entropy(y_pred, y_true)
loss


Value: 0.6936219317038546

In [19]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=100,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2
)

X.shape, y.shape

((100, 20), (100,))

In [22]:
model = Sequential(
    Linear(20, 32),
    ReLU(),
    Linear(32, 1),
    Sigmoid()
)

In [ ]:
# Training Loop
BATCH_SIZE = 32
EPOCHS = 10
lr = 0.001
for epoch in range(EPOCHS):
    data = list(zip(X, y))
    random.shuffle(data)
    X, y = zip(*data)
    X = list(X)
    y = list(y)
    for i in range(0, len(X), BATCH_SIZE):
        X_batch = X[i: i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]
        losses = []
        for x_i, y_i in zip(X_batch, y_batch):
            y_pred = model(x_i)
            loss = binary_cross_entropy(y_pred,y_i)
            losses.append(loss)
        loss = sum(losses) * (1 / len(losses))
        model.zero_grad()
        loss.backward()
        for p in model.parameters():
            p.data -= lr * p.grad
    
    print(epoch, loss.data)
            

0 0.7177515625897952
1 0.7252847916374313
2 0.6898295313232925
3 0.6789978719088293
4 0.7517479119676136
5 0.6516488059447507
6 0.7028571418704689
7 0.6849760733920164
8 0.7168115505470946
9 0.639872706441651
